# 03 · 少样本 AI 风格微调（SDT + LoRA）、全字集多异形生成与 `.hwfont` 导出

本 Notebook 完成字体构建最后阶段：
1. 载入 Notebook 02 生成的真实书写者单线轨迹 `02_trajectories.json`；
2. 在 Colab GPU（T4 / L4 / A100）上启用 **SDT（Style-Disentangled Transformer）+ LoRA** 少样本微调（若无 GPU 则自动无缝切换至本地参数化风格迁移引擎 `LocalStyleTransferAugmentor`）；
3. 为目标字表批量生成每字 **4 个同字异形**（真实扫描样本优先保留 `source="scan", weight=1.5`）；
4. 执行自动化质检门禁（点密度、外接框合理性、异形 Chamfer 多样性 `>= 18u`、可选 OCR 回检）并通过 `HwFontWriter` + `HwFontReader.verify()` 导出可直接导入本地上位机的单文件 `.hwfont`。

In [ ]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path
import numpy as np

IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install torch numpy pillow pydantic

REPO_ROOT = Path("/content/inkcraft") if IN_COLAB and Path("/content/inkcraft").exists() else Path.cwd().parents[1]
for sub in ("packages/hwfont_sdk", "colab_builder", "backend"):
    p = str(REPO_ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

WRITER_NAME = "zhangsan"
WRITER_DISPLAY_NAME = "张三"
WRITER_ALIAS = "行楷·个人笔迹"
WORK_ROOT = Path("/content/drive/MyDrive/inkcraft_builder") / WRITER_NAME if IN_COLAB else Path("./work") / WRITER_NAME
TRAJ_PATH = WORK_ROOT / "02_trajectories.json"
OUT_HWFONT = WORK_ROOT / f"{WRITER_NAME}_v1.0.hwfont"
print("输出字体目标:", OUT_HWFONT)

## 1. 载入真实轨迹并训练风格模型（GPU SDT-LoRA / CPU 参数化风格学习）

In [ ]:
from hwbuilder.model_sdt import (
    LocalStyleTransferAugmentor,
    SDTConfig,
    SDTTrainer,
    is_torch_available,
)
from hwbuilder.skeleton_align import load_builtin_priors, load_makemeahanzi_priors

priors = load_builtin_priors()
MAKEMEAHANZI_PATH = WORK_ROOT.parent / "priors" / "graphics.txt"
if MAKEMEAHANZI_PATH.exists():
    priors.update(load_makemeahanzi_priors(MAKEMEAHANZI_PATH))

records = json.loads(TRAJ_PATH.read_text(encoding="utf-8")) if TRAJ_PATH.exists() else []
scanned_by_char = defaultdict(list)
for rec in records:
    strokes = [np.asarray(s, dtype=np.int16) for s in rec["strokes"]]
    scanned_by_char[rec["char"]].append((strokes, float(rec.get("quality", 0.9))))

# 1) 始终拟合参数化风格画像（用于快速兜底与多异形微调）
augmentor = LocalStyleTransferAugmentor()
style_profile = augmentor.fit(
    {ch: [v[0] for v in vs] for ch, vs in scanned_by_char.items()},
    writer_name=WRITER_NAME,
)
style_profile.save(WORK_ROOT / "writer_style_profile.json")
print("学习到的书写风格画像:", style_profile)

# 2) 在 Colab GPU 上启用 SDT + LoRA 少样本微调
USE_NEURAL_SDT = is_torch_available() and len(scanned_by_char) >= 16
sdt_trainer = None
if USE_NEURAL_SDT:
    pairs = [
        (priors[ch], variants[0][0])
        for ch, variants in scanned_by_char.items()
        if ch in priors and variants
    ]
    if len(pairs) >= 8:
        sdt_cfg = SDTConfig(epochs=60, batch_size=16, lora_rank=8, lr=2e-4)
        sdt_trainer = SDTTrainer(sdt_cfg)
        history = sdt_trainer.fit(pairs, lora_only=True)
        sdt_trainer.save_checkpoint(WORK_ROOT / "sdt_lora_ckpt.pt")
        print(f"SDT-LoRA 微调完成 ({len(pairs)} 配对样本)，末轮 loss={history['loss'][-1]:.4f}")
    else:
        print("配对先验样本不足，跳过 SDT 神经微调，使用参数化风格迁移引擎。")
else:
    print("当前为 CPU 或小样本模式，使用 LocalStyleTransferAugmentor 生成多异形。")

## 2. 批量生成同字异形、质检门禁并导出 `.hwfont`

导出的 `.hwfont` 为标准单文件 SQLite，已执行 `content_sha256` 完整性签名与 `VACUUM` 压缩，下载后直接拖入本地上位机 `data/writers/` 或通过 Web 界面导入即可热插拔使用。

In [ ]:
from hwfont_sdk import HwFontMeta
from hwbuilder.qc_export import QCConfig, VariantCandidate, export_hwfont_detailed

TARGET_VARIANTS_PER_CHAR = 4
char_variants = {}

# 合并真实扫描字集 + 先验目标字集
target_chars = list(dict.fromkeys(list(scanned_by_char.keys()) + list(priors.keys())))

for ch in target_chars:
    cands = []
    # 真实扫描轨迹优先入库（高权重）
    for strokes, q in scanned_by_char.get(ch, []):
        cands.append(VariantCandidate(strokes=strokes, source="scan", quality=q, weight=1.5))

    # 补充生成异形至目标数量（多采 2 个候选以留给 Chamfer 多样性门禁筛选）
    need = max(0, TARGET_VARIANTS_PER_CHAR - len(cands))
    if need > 0:
        base = cands[0].strokes if cands else priors.get(ch)
        if base:
            gen_list = augmentor.synthesize_variants(
                ch,
                base,
                n_variants=need + 2,
                seed=2026,
                diversity_boost=1.15,
            )
            for g_strokes in gen_list:
                cands.append(VariantCandidate(strokes=g_strokes, source="generated", quality=0.92, weight=1.0))
    if cands:
        char_variants[ch] = cands

meta = HwFontMeta(
    writer_name=WRITER_DISPLAY_NAME,
    writer_alias=WRITER_ALIAS,
    author="HWBuilder Colab Pipeline",
    version="1.0.0",
    style_tags=["行楷", "Colab构建"],
    builder="hwbuilder 1.0.0 (colab)",
    model="sdt-lora-r8" if sdt_trainer is not None else "local-style-transfer-v1",
    license=f"Authorized personal handwriting font of {WRITER_DISPLAY_NAME}",
)

qc_cfg = QCConfig(min_chamfer_diversity_u=18.0, embed_thumbnails=False)
result = export_hwfont_detailed(meta, char_variants, OUT_HWFONT, qc_config=qc_cfg, priors=priors)

print("导出路径:", result.out_path)
print("校验结果 ok:", result.verify_report.ok)
print(f"写入字符: {result.total_chars}, 通过异形: {result.accepted_variants}, 质检拦截: {result.rejected_variants}")
print("content_sha256:", result.verify_report.actual_sha256)
assert result.verify_report.ok, f"Verification failed: {result.verify_report.errors}"

In [ ]:
# 在 Colab 中自动触发浏览器下载 .hwfont 文件
if IN_COLAB and OUT_HWFONT.exists():
    from google.colab import files
    files.download(str(OUT_HWFONT))